# Cellpose-SAM: 1050-nm red-channel session masks

This notebook follows the multi-mouse project layout:

`molecular_tracking_derivatives/<mouse>/sessions/<YYYYMMDD>/1050/preprocessing/red.tif`

and writes the labeled Cellpose mask to:

`molecular_tracking_derivatives/<mouse>/sessions/<YYYYMMDD>/1050/segmentation/mask.tif`

Only the 1050-nm red channel is segmented. Existing `mask.tif` files are skipped by default.

In [29]:
from pathlib import Path
import csv
import os
import uuid

import numpy as np
import tifffile
from tqdm.auto import tqdm
from cellpose import models
from cellpose.io import imread_3D

In [32]:
# ------------------------------------------------------------------
# Settings
# ------------------------------------------------------------------
DERIVATIVES_ROOT = Path(r"D:\_data\_newAAV_2026\molecular_tracking_derivatives")

# None = process every mouse that has session-based 1050 red images.
# Or use, for example:
# MOUSE_IDS = ["Fucci-Dead_1", "Fucci-Dead_2", "Fucci-Tri_1", "Fucci-Tri_2"]
MOUSE_IDS = None
MOUSE_IDS = [
    "Fucci-Dead_1",
    "Fucci-Dead_2",
    "Fucci-Tri_3",
]

OVERWRITE_EXISTING = False
MIN_SIZE = 100

if not DERIVATIVES_ROOT.is_dir():
    raise FileNotFoundError(f"Derivatives root was not found: {DERIVATIVES_ROOT}")

In [33]:
# Discover only:
# <mouse>/sessions/<YYYYMMDD>/1050/preprocessing/red.tif
red_paths = []

mouse_dirs = sorted(
    p for p in DERIVATIVES_ROOT.iterdir()
    if p.is_dir() and not p.name.startswith("_")
)

if MOUSE_IDS is not None:
    wanted = set(MOUSE_IDS)
    mouse_dirs = [p for p in mouse_dirs if p.name in wanted]

for mouse_dir in mouse_dirs:
    sessions_dir = mouse_dir / "sessions"
    if not sessions_dir.is_dir():
        continue

    for session_dir in sorted(p for p in sessions_dir.iterdir() if p.is_dir()):
        red_path = session_dir / "1050" / "preprocessing" / "red.tif"
        if red_path.is_file():
            red_paths.append(red_path)

if not red_paths:
    raise FileNotFoundError(
        "No session-based 1050 red.tif files were found beneath:\n"
        f"{DERIVATIVES_ROOT}"
    )

print(f"Found {len(red_paths)} 1050-nm red stack(s):")
for path in red_paths:
    mouse_id = path.parents[4].name
    session_id = path.parents[2].name
    print(f"  {mouse_id} / {session_id} -> {path}")

Found 23 1050-nm red stack(s):
  Fucci-Dead_1 / 20260819 -> D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Dead_1\sessions\20260819\1050\preprocessing\red.tif
  Fucci-Dead_1 / 20260820 -> D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Dead_1\sessions\20260820\1050\preprocessing\red.tif
  Fucci-Dead_1 / 20260821 -> D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Dead_1\sessions\20260821\1050\preprocessing\red.tif
  Fucci-Dead_1 / 20260824 -> D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Dead_1\sessions\20260824\1050\preprocessing\red.tif
  Fucci-Dead_1 / 20260825 -> D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Dead_1\sessions\20260825\1050\preprocessing\red.tif
  Fucci-Dead_1 / 20260827 -> D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Dead_1\sessions\20260827\1050\preprocessing\red.tif
  Fucci-Dead_1 / 20260828 -> D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Dead_1\sessions\20260828\1050\preprocessi

In [34]:
# Same Cellpose-SAM model/settings used by the previous notebook.
model = models.CellposeModel(
    gpu=True,
    pretrained_model="cpsam_v2",
)

In [35]:
results = []

for red_path in tqdm(red_paths, desc="Cellpose-SAM 1050 sessions"):
    mouse_id = red_path.parents[4].name
    session_id = red_path.parents[2].name

    segmentation_dir = red_path.parent.parent / "segmentation"
    output_path = segmentation_dir / "mask.tif"

    if output_path.exists() and not OVERWRITE_EXISTING:
        tqdm.write(f"Skipping existing: {mouse_id} {session_id} -> {output_path}")
        results.append({
            "mouse_id": mouse_id,
            "session_id": session_id,
            "red_path": str(red_path),
            "mask_path": str(output_path),
            "status": "ALREADY_EXISTS",
            "n_masks": "",
            "error": "",
        })
        continue

    temp_path = None

    try:
        tqdm.write(f"Processing: {mouse_id} {session_id}")

        loaded_image = imread_3D(red_path.as_posix())

        masks, flows, styles = model.eval(
            loaded_image,
            do_3D=True,
            z_axis=0,
            channel_axis=3,
            min_size=MIN_SIZE,
        )

        n_masks = int(np.max(masks))
        tqdm.write(f"  Found {n_masks} mask(s)")

        segmentation_dir.mkdir(parents=True, exist_ok=True)

        # Store the labeled volume directly as the pipeline's canonical mask.tif.
        # uint16 is sufficient unless there are >65,535 labels.
        mask_dtype = np.uint16 if n_masks <= np.iinfo(np.uint16).max else np.uint32
        mask_to_save = masks.astype(mask_dtype, copy=False)

        temp_path = segmentation_dir / f".mask.tif.tmp.{uuid.uuid4().hex}"

        tifffile.imwrite(
            temp_path,
            mask_to_save,
            photometric="minisblack",
            metadata={"axes": "ZYX"},
        )

        # Read the temporary file back before promotion.
        check = tifffile.imread(temp_path)
        if check.shape != mask_to_save.shape:
            raise RuntimeError(
                f"Saved mask shape mismatch: expected {mask_to_save.shape}, got {check.shape}"
            )

        if output_path.exists():
            raise FileExistsError(
                f"Destination appeared during processing; refusing to overwrite: {output_path}"
            )

        # On Windows os.rename refuses to replace an existing destination.
        os.rename(temp_path, output_path)
        temp_path = None

        results.append({
            "mouse_id": mouse_id,
            "session_id": session_id,
            "red_path": str(red_path),
            "mask_path": str(output_path),
            "status": "SEGMENTED",
            "n_masks": n_masks,
            "error": "",
        })

        tqdm.write(f"Saved: {output_path}")

    except Exception as error:
        if temp_path is not None and temp_path.exists():
            temp_path.unlink()

        results.append({
            "mouse_id": mouse_id,
            "session_id": session_id,
            "red_path": str(red_path),
            "mask_path": str(output_path),
            "status": "FAILED",
            "n_masks": "",
            "error": str(error),
        })

        tqdm.write(f"FAILED: {mouse_id} {session_id}")
        tqdm.write(f"  {error}")

Cellpose-SAM 1050 sessions:   0%|          | 0/23 [00:00<?, ?it/s]

Skipping existing: Fucci-Dead_1 20260819 -> D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Dead_1\sessions\20260819\1050\segmentation\mask.tif
Skipping existing: Fucci-Dead_1 20260820 -> D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Dead_1\sessions\20260820\1050\segmentation\mask.tif
Skipping existing: Fucci-Dead_1 20260821 -> D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Dead_1\sessions\20260821\1050\segmentation\mask.tif
Skipping existing: Fucci-Dead_1 20260824 -> D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Dead_1\sessions\20260824\1050\segmentation\mask.tif
Skipping existing: Fucci-Dead_1 20260825 -> D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Dead_1\sessions\20260825\1050\segmentation\mask.tif
Skipping existing: Fucci-Dead_1 20260827 -> D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Dead_1\sessions\20260827\1050\segmentation\mask.tif
Skipping existing: Fucci-Dead_1 20260828 -> D:\_data\_newAAV_2026\mole

Cellpose-SAM 1050 sessions: 100%|██████████| 23/23 [14:22<00:00, 37.48s/it]

  Found 2383 mask(s)
Saved: D:\_data\_newAAV_2026\molecular_tracking_derivatives\Fucci-Tri_3\sessions\20260904\1050\segmentation\mask.tif


In [36]:
log_path = DERIVATIVES_ROOT / "cellposeSAM_1050_session_log.csv"

with log_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=[
            "mouse_id",
            "session_id",
            "red_path",
            "mask_path",
            "status",
            "n_masks",
            "error",
        ],
    )
    writer.writeheader()
    writer.writerows(results)

print("\nFinished.")
print(f"Log: {log_path}")

status_counts = {}
for row in results:
    status_counts[row["status"]] = status_counts.get(row["status"], 0) + 1

for status, count in sorted(status_counts.items()):
    print(f"{status:20s} {count:4d}")

failed = [row for row in results if row["status"] == "FAILED"]
if failed:
    print("\nFailed sessions:")
    for row in failed:
        print(f"  {row['mouse_id']} {row['session_id']}: {row['error']}")


Finished.
Log: D:\_data\_newAAV_2026\molecular_tracking_derivatives\cellposeSAM_1050_session_log.csv
ALREADY_EXISTS         22
SEGMENTED               1
